# Gene Trajectory GNN

Модель для контекст-управляемого поиска генных программ.

**Архитектура:**
- **PPGN** (WL-3) на статическом NEKO-графе → структурный эмбеддинг
- **GAT** на hdWGCNA-графе per timepoint → динамический эмбеддинг
- **OT-выравнивание** (Wasserstein Procrustes) между timepoint-ами
- **Trajectory head**: delta + mean → финальный эмбеддинг для ретривела

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from typing import List, Optional

print("torch:", torch.__version__)
print("torch_geometric: ok")

## 1. PPGN — Provably Powerful Graph Network (WL-3)\n\nПрименяется к **статическому NEKO-графу** (механистические рёбра из баз данных).\n\nКлючевая операция: `X_prod[i,j] = Σ_k X[i,k] · X[k,j]` — матричное произведение по фиче-оси.\nЭто захватывает пути длины 2 (треугольники, петли обратной связи) — источник WL-3 выразительности.\n\nТензор `X[n×n×d]`: `X[i,i]` = фичи узла i, `X[i,j]` = фичи ребра (i,j).

In [ ]:
class PPGNLayer(nn.Module):
    """
    Один слой PPGN на тензоре n×n×d.

    X_new = MLP([X, row_sum, col_sum, X@X, diag_broadcast])
    X@X — матричное произведение → пути длины 2 → WL-3 мощность.
    """

    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim * 5, out_dim * 2),
            nn.GELU(),
            nn.Linear(out_dim * 2, out_dim),
        )
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        """X : [n, n, d]"""
        n, _, d = X.shape

        row_sum = X.sum(dim=1, keepdim=True).expand(n, n, d)   # [n,n,d]
        col_sum = X.sum(dim=0, keepdim=True).expand(n, n, d)   # [n,n,d]

        # WL-3: X_prod[i,j] = sum_k X[i,k] * X[k,j]
        X_prod  = torch.einsum('ikd,kjd->ijd', X, X)           # [n,n,d]

        # Диагональ (фичи узла) → broadcast на все пары
        diag    = torch.diagonal(X, dim1=0, dim2=1).T          # [n, d]
        diag_bc = diag.unsqueeze(1).expand(n, n, d)            # [n,n,d]

        X_cat = torch.cat([X, row_sum, col_sum, X_prod, diag_bc], dim=-1)
        return self.norm(self.mlp(X_cat))


class PPGN(nn.Module):
    """
    Полный PPGN-энкодер для NEKO-графа.

    2 слоя достаточно для WL-3:
      слой 1 — пути длины 2 (X@X захватывает треугольники и петли)
      слой 2 — пути длины 4 и замкнутые регуляторные мотивы через двойной einsum

    Вход:  node_feats [n, node_dim]  +  adj [n, n]
    Выход: node embeddings [n, out_dim]
    """

    def __init__(self, node_dim: int, hidden_dim: int,
                 out_dim: int, n_layers: int = 2):
        super().__init__()
        self.node_proj = nn.Linear(node_dim, hidden_dim)
        self.layers    = nn.ModuleList(
            [PPGNLayer(hidden_dim, hidden_dim) for _ in range(n_layers)]
        )
        self.readout = nn.Sequential(
            nn.Linear(hidden_dim, out_dim),
            nn.GELU(),
            nn.Linear(out_dim, out_dim),
        )

    def forward(self, node_feats: torch.Tensor,
                adj: torch.Tensor) -> torch.Tensor:
        """
        node_feats : [n, node_dim]
        adj        : [n, n]  взвешенная матрица смежности NEKO
        """
        n = node_feats.shape[0]
        h = self.node_proj(node_feats)                          # [n, hidden_dim]

        # Начальный пайрвайз-тензор:
        #   X[i,j] = adj[i,j] * h[j]   (взвешенные фичи соседа)
        #   X[i,i] = h[i]               (фичи самого узла)
        X   = adj.unsqueeze(-1) * h.unsqueeze(0)               # [n, n, hidden_dim]
        idx = torch.arange(n, device=node_feats.device)
        X[idx, idx] = h

        for layer in self.layers:
            X = X + layer(X)                                    # residual

        node_emb = X[idx, idx]                                  # [n, hidden_dim]
        return self.readout(node_emb)                           # [n, out_dim]


print("PPGN defined")

## 2. GAT Encoder

GAT применяется к **hdWGCNA-графу** отдельно для каждого timepoint-а.

Время уже закодировано в самом графе — `graph_t0` и `graph_t80` структурно разные  
(разные рёбра, разные веса Pearson) потому что строились на разных клетках.  
GAT с shared weights автоматически даёт разные эмбеддинги для разных графов.  
Дополнительное time encoding было бы двойным счётом.

Attention mechanism down-weightит шумные рёбра WGCNA — решает проблему  
«не все коэкспрессионные связи биологически значимы».

In [ ]:
class GATEncoder(nn.Module):
    """
    Многослойный GAT на hdWGCNA-графе одного timepoint-а.
    Shared weights между timepoint-ами — время закодировано в структуре графа.
    """

    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int,
                 n_heads: int = 4, n_layers: int = 3, dropout: float = 0.1):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.dropout    = nn.Dropout(dropout)

        self.gat_layers = nn.ModuleList()
        self.norms      = nn.ModuleList()

        for i in range(n_layers):
            is_last = (i == n_layers - 1)
            in_ch   = hidden_dim * n_heads if i > 0 else hidden_dim
            heads   = 1 if is_last else n_heads
            out_ch  = out_dim if is_last else hidden_dim
            concat  = not is_last

            self.gat_layers.append(
                GATConv(in_ch, out_ch, heads=heads,
                        dropout=dropout, concat=concat)
            )
            self.norms.append(
                nn.LayerNorm(out_ch * heads if concat else out_ch)
            )

    def forward(self, x: torch.Tensor,
                edge_index: torch.Tensor,
                edge_attr: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        x          : [n_genes, in_dim]
        edge_index : [2, n_edges]   hdWGCNA рёбра этого timepoint-а
        edge_attr  : [n_edges, 1]   веса рёбер (Pearson r^beta)
        """
        x = self.input_proj(x)

        for layer, norm in zip(self.gat_layers, self.norms):
            x_in = x
            x    = layer(x, edge_index)
            x    = norm(F.gelu(x))
            x    = self.dropout(x)
            if x.shape == x_in.shape:
                x = x + x_in

        return x


print("GATEncoder defined (no time encoding — time is in the graph)")

## 3. Temporal delta — прямое сравнение эмбеддингов

GAT использует **shared weights** → все timepoint-ы живут в одном пространстве `R^{emb_dim}` по построению.  
Выравнивание не нужно — delta считается напрямую по индексу гена:

```
delta[i] = emb_tN[i] - emb_t0[i]
```

Биологический смысл: насколько изменилось коэкспрессионное окружение гена i  
в WGCNA-графе от первого до последнего timepoint-а.

**mean** = средняя позиция гена в пространстве за весь эксперимент → стабильные соседи.  
**delta** = направление и величина сдвига → гены которые меняют контекст (например FOXM1 при пролиферации).

In [ ]:
import ot   # pip install POT

class OTAlignment(nn.Module):
    """
    Оптимальный транспорт между облаками генных эмбеддингов двух timepoint-ов.

    Почему OT, а не прямой delta:
    - GAT агрегирует статистику соседства, не кодирует идентичность соседей
    - Ген i при t=0 и ген i при t=80 имеют разное коэкспрессионное окружение
      (разные клетки, разные состояния) → частичная анонимность
    - OT находит оптимальное соответствие между облаками без предположения
      что i → i

    Sinkhorn (энтропийно-регуляризованный OT):
      min_{P}  <C, P>  +  ε * KL(P || ab^T)
      C[i,j] = ||source[i] - target[j]||²

    Барицентрическая проекция:
      aligned[i] = Σ_j P[i,j] * target[j] / Σ_j P[i,j]
    """

    def __init__(self, reg: float = 0.05, n_iter: int = 100):
        super().__init__()
        self.reg    = reg
        self.n_iter = n_iter

    @torch.no_grad()
    def align_pair(self, source: torch.Tensor,
                   target: torch.Tensor) -> torch.Tensor:
        n = source.shape[0]
        a = torch.ones(n) / n
        b = torch.ones(n) / n

        C = torch.cdist(source, target).pow(2)

        P = ot.sinkhorn(
            a.numpy(), b.numpy(),
            C.numpy(),
            reg       = self.reg,
            numItermax= self.n_iter,
            stopThr   = 1e-6,
            warn      = False,
        )
        P = torch.tensor(P, dtype=source.dtype)

        # Барицентрическая проекция
        row_sums = P.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return (P @ target) / row_sums   # [n, dim]

    def align_sequence(self, embs: List[torch.Tensor]) -> List[torch.Tensor]:
        """Выравниваем все timepoint-ы к t=0."""
        ref     = embs[0]
        aligned = [ref]
        for emb in embs[1:]:
            aligned.append(self.align_pair(emb, ref))
        return aligned


print("OTAlignment defined (Sinkhorn + barycentric projection)")

## 4. GeneTrajectoryModel — главная модель\n\nПосле фильтрации осталось 3887 генов — PPGN на таком количестве требует ~77 GB (не влезает).\n\n**Архитектура:**\n```\nВсе 3887 генов:\n  GAT(hdWGCNA_t) → emb_t  [n, emb_dim]   per timepoint\n  OT-align → delta + mean → trajectory_emb\n\nГены программы (~200, отдельно):\n  PPGN(NEKO) → structural_emb             per program query\n```\n\n`GeneTrajectoryModel` — только GAT + OT. PPGN вызывается отдельно через `ProgramEncoder`.

In [ ]:
class GeneTrajectoryModel(nn.Module):
    """
    GAT per timepoint + OT alignment.

    Время → в структуре WGCNA-графов (разные рёбра/веса per timepoint).
    Гены частично анонимны (GAT кодирует структуру соседства) → OT.
    """

    def __init__(
        self,
        gene_feat_dim: int = 3,
        hidden_dim: int    = 128,
        emb_dim: int       = 128,
        gat_layers: int    = 3,
        gat_heads: int     = 4,
        dropout: float     = 0.1,
        ot_reg: float      = 0.05,
        ot_iter: int       = 100,
    ):
        super().__init__()
        self.gat = GATEncoder(
            in_dim=gene_feat_dim, hidden_dim=hidden_dim,
            out_dim=emb_dim, n_heads=gat_heads,
            n_layers=gat_layers, dropout=dropout,
        )
        self.ot_align  = OTAlignment(reg=ot_reg, n_iter=ot_iter)
        self.traj_head = nn.Sequential(
            nn.Linear(emb_dim * 2, emb_dim),
            nn.GELU(),
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, emb_dim),
        )

    def forward(self, gene_feats_per_t, hdwgcna_edges_per_t,
                hdwgcna_attrs_per_t=None):

        T     = len(gene_feats_per_t)
        attrs = hdwgcna_attrs_per_t or [None] * T

        # 1. GAT — время в графе, не в модели
        snapshots = [
            self.gat(gene_feats_per_t[t], hdwgcna_edges_per_t[t], attrs[t])
            for t in range(T)
        ]

        # 2. OT — выравниваем облака к t=0
        aligned  = self.ot_align.align_sequence(snapshots)

        stacked  = torch.stack(aligned, dim=0)
        emb_mean = stacked.mean(dim=0)
        delta    = aligned[-1] - aligned[0]

        trajectory_emb = self.traj_head(
            torch.cat([delta, emb_mean], dim=-1)
        )

        return {
            "trajectory_emb": trajectory_emb,
            "snapshots":      snapshots,
            "aligned":        aligned,
            "delta":          delta,
        }


class ProgramEncoder(nn.Module):
    """PPGN для конкретной программы (~200 генов) на NEKO-графе."""

    def __init__(self, gene_feat_dim=3, hidden_dim=64,
                 out_dim=128, ppgn_layers=2):
        super().__init__()
        self.ppgn = PPGN(node_dim=gene_feat_dim, hidden_dim=hidden_dim,
                         out_dim=out_dim, n_layers=ppgn_layers)

    def forward(self, gene_feats, neko_adj):
        return self.ppgn(gene_feats, neko_adj)


print("GeneTrajectoryModel (GAT + OT) + ProgramEncoder defined")

## 5. Loss — трёхкомпонентная функция потерь\n\n| Компонент | Сигнал | λ |\n|---|---|---|\n| `L_coexpr` | hdWGCNA-соседи → близко в trajectory space | 1.0 |\n| `L_neko` | NEKO-пары → близко (мягкий механистический приор) | 0.3 |\n| `L_temporal` | Консекутивные aligned-эмбеддинги → плавные | 0.2 |\n\nInfoNCE: позитивные пары должны быть ближе, чем все остальные гены в батче.

In [ ]:
class TrajectoryLoss(nn.Module):
    """
    L = L_coexpr  +  λ_neko * L_neko  +  λ_temporal * L_smooth

    L_coexpr   — InfoNCE: hdWGCNA-соседи → близко в trajectory space
    L_neko     — InfoNCE: NEKO-пары → близко (λ < 1, мягкий приор)
    L_temporal — MSE: плавность между последовательными aligned-эмбеддингами
    """

    def __init__(self, lambda_neko: float = 0.3,
                 lambda_temporal: float = 0.2,
                 temperature: float = 0.07):
        super().__init__()
        self.lambda_neko     = lambda_neko
        self.lambda_temporal = lambda_temporal
        self.T               = temperature

    def infonce(self, emb: torch.Tensor,
                pos_pairs: torch.Tensor) -> torch.Tensor:
        """
        emb       : [n, d]   эмбеддинги генов
        pos_pairs : [m, 2]   индексы позитивных пар (от hdWGCNA или NEKO)
        """
        emb     = F.normalize(emb, dim=-1)
        pos_sim = (emb[pos_pairs[:, 0]] * emb[pos_pairs[:, 1]]).sum(-1) / self.T
        all_sim = emb @ emb.T / self.T
        loss    = -pos_sim + torch.logsumexp(all_sim, dim=-1)[pos_pairs[:, 0]]
        return loss.mean()

    def temporal_smoothness(self, aligned: List[torch.Tensor]) -> torch.Tensor:
        """Штрафует большие скачки между последовательными timepoint-ами."""
        diffs = [
            (aligned[t + 1] - aligned[t]).pow(2).mean()
            for t in range(len(aligned) - 1)
        ]
        return torch.stack(diffs).mean()

    def forward(self, outputs: dict,
                hdwgcna_pos_pairs: torch.Tensor,
                neko_pos_pairs: torch.Tensor) -> dict:

        emb     = outputs["trajectory_emb"]
        aligned = outputs["aligned"]

        L_coexpr   = self.infonce(emb, hdwgcna_pos_pairs)
        L_neko     = self.infonce(emb, neko_pos_pairs)
        L_temporal = self.temporal_smoothness(aligned)

        total = (L_coexpr
                 + self.lambda_neko     * L_neko
                 + self.lambda_temporal * L_temporal)

        return {
            "loss":          total,
            "loss_coexpr":   L_coexpr,
            "loss_neko":     L_neko,
            "loss_temporal": L_temporal,
        }


print("TrajectoryLoss defined")

## 6. Sanity Check\n\n200 генов, 5 timepoint-ов, dummy данные. Проверяем формы и что лосс конечный.

In [ ]:
N = 200; T = 5; D = 16

model   = GeneTrajectoryModel(gene_feat_dim=D, hidden_dim=128, emb_dim=128)
loss_fn = TrajectoryLoss()

gene_feats_per_t    = [torch.randn(N, D) for _ in range(T)]
neko_adj            = (torch.rand(N, N) > 0.9).float()
neko_adj            = (neko_adj + neko_adj.T).clamp(0, 1)
hdwgcna_edges_per_t = [torch.randint(0, N, (2, 600)) for _ in range(T)]

out = model(gene_feats_per_t, hdwgcna_edges_per_t)

print("trajectory_emb :", out["trajectory_emb"].shape)
print("delta          :", out["delta"].shape)
print("n_snapshots    :", len(out["snapshots"]))

pos_pairs = torch.randint(0, N, (80, 2))
losses    = loss_fn(out, pos_pairs, pos_pairs)
print(f"\ntotal loss : {losses['loss'].item():.4f}")

prog_encoder = ProgramEncoder(gene_feat_dim=D)
prog_emb = prog_encoder(gene_feats_per_t[0], neko_adj)
print(f"ProgramEncoder : {prog_emb.shape}")

---\n## 7. Загрузка данных\n\n`data_uncut.h5ad` — 13 968 клеток × 8 442 гена, 6 timepoint-ов (t=0,16,32,48,64,80), 3 типа клеток.

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

DATA_PATH = "/Users/zeev/CardamomOT/my_project/Data/data_uncut.h5ad"

adata = sc.read_h5ad(DATA_PATH)
print(f"shape       : {adata.shape}")
print(f"timepoints  : {sorted(adata.obs['time'].unique())}")
print(f"cell types  : {sorted(adata.obs['cell_type'].unique())}")
print(f"cells/tp    : {adata.obs['time'].value_counts().sort_index().to_dict()}")

## 8. Препроцессинг и отбор генов\n\nВместо «топ N HVG» — убираем гены которые **точно не несут информации**:\n- mean expression < 0.1 → ген почти не экспрессируется\n- std < 0.1 → ген не меняется между клетками (flat)\n- detection rate < 5% → ген есть менее чем в 5% клеток\n\nВсё что прошло фильтр — идёт в сеть. Количество не фиксируем заранее.

In [ ]:
import scipy.sparse

# Нормализация (raw counts → CPM → log1p)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Считаем статистики по всем клеткам
X_all = adata.X
if scipy.sparse.issparse(X_all):
    X_all = X_all.toarray()

mean_expr = X_all.mean(axis=0)          # [n_genes]
std_expr  = X_all.std(axis=0)           # [n_genes]
detection = (X_all > 0).mean(axis=0)   # [n_genes]  доля ненулевых клеток

# Пороги — убираем биологически пустые гены
MIN_MEAN      = 0.1    # минимальная средняя экспрессия
MIN_STD       = 0.1    # минимальная дисперсия (flat гены)
MIN_DETECTION = 0.05   # ген должен быть в ≥5% клеток

keep = (
    (mean_expr  >= MIN_MEAN)      &
    (std_expr   >= MIN_STD)       &
    (detection  >= MIN_DETECTION)
)

gene_names = adata.var_names[keep].tolist()
adata_hvg  = adata[:, keep].copy()

print(f"Всего генов     : {adata.shape[1]}")
print(f"Прошло фильтр   : {keep.sum()}  ({keep.mean()*100:.1f}%)")
print(f"  mean ≥ {MIN_MEAN}   : {(mean_expr  >= MIN_MEAN).sum()}")
print(f"  std  ≥ {MIN_STD}   : {(std_expr   >= MIN_STD).sum()}")
print(f"  det  ≥ {MIN_DETECTION}  : {(detection  >= MIN_DETECTION).sum()}")
print(f"\nРабочий объект  : {adata_hvg.shape}")

# Предупреждение если слишком много генов для PPGN
N_FILTERED = keep.sum()
if N_FILTERED > 3000:
    print(f"\n⚠️  {N_FILTERED} генов > 3000 — PPGN будет тяжёлым ({N_FILTERED**2*128*4/1e9:.1f} GB)")
    print("   GAT работает нормально. PPGN запускай только на генах программы (~200).")
else:
    mem_gb = N_FILTERED**2 * 128 * 4 / 1e9
    print(f"\n✓  PPGN тензор: {N_FILTERED}×{N_FILTERED}×128 ≈ {mem_gb:.2f} GB")

## 9. Blockwise WGCNA per timepoint\n\nАдаптация твоего WGCNA-кода для каждого timepoint-а отдельно.\n\n- Стандартизация генов (z-score) перед корреляцией\n- Blockwise Pearson (memory-safe для 2000 генов)\n- Soft thresholding β=6\n- Результат: `edge_index` + `edge_attr` для GAT (без NetworkX)\n\n> Метаклетки не нужны — 2328 клеток на timepoint достаточно для стабильных корреляций.

In [ ]:
import scipy.sparse
from tqdm import tqdm

BETA        = 6      # soft threshold power
THRESHOLD   = 0.01   # adjacency threshold после soft thresholding
CHUNK_SIZE  = 500    # blockwise chunk (memory-safe)

timepoints = sorted(adata_hvg.obs['time'].unique())
print(f"Timepoints: {timepoints}")


def build_wgcna_per_timepoint(X_t: np.ndarray):
    """
    X_t : [n_cells, n_genes]  — уже нормализованная и log1p матрица

    1. Стандартизация генов (z-score)
    2. Blockwise Pearson correlation
    3. Soft thresholding |corr|^beta
    4. Порог — убираем слабые рёбра
    5. Возвращает edge_index [2, E] и edge_attr [E, 1]
    """
    n_cells, n_genes = X_t.shape

    # Z-score по генам
    X_std = (X_t - X_t.mean(axis=0)) / (X_t.std(axis=0) + 1e-8)

    rows_all, cols_all, weights_all = [], [], []

    for start_i in range(0, n_genes, CHUNK_SIZE):
        end_i = min(start_i + CHUNK_SIZE, n_genes)
        Xi = X_std[:, start_i:end_i]

        for start_j in range(start_i, n_genes, CHUNK_SIZE):
            end_j = min(start_j + CHUNK_SIZE, n_genes)
            Xj = X_std[:, start_j:end_j]

            corr_block = (Xi.T @ Xj) / (n_cells - 1)
            adj_block  = np.abs(corr_block) ** BETA

            r_idx, c_idx = np.where(adj_block >= THRESHOLD)
            for r, c in zip(r_idx, c_idx):
                gi = start_i + r
                gj = start_j + c
                if gi >= gj:
                    continue
                rows_all.append(gi)
                cols_all.append(gj)
                weights_all.append(adj_block[r, c])

    if len(rows_all) == 0:
        edge_index = torch.zeros(2, 0, dtype=torch.long)
        edge_attr  = torch.zeros(0, 1, dtype=torch.float32)
        return edge_index, edge_attr

    rows_all    = np.array(rows_all)
    cols_all    = np.array(cols_all)
    weights_all = np.array(weights_all, dtype=np.float32)

    # Неориентированный граф
    rows_bi    = np.concatenate([rows_all, cols_all])
    cols_bi    = np.concatenate([cols_all, rows_all])
    weights_bi = np.concatenate([weights_all, weights_all])

    edge_index = torch.tensor(np.stack([rows_bi, cols_bi]), dtype=torch.long)
    edge_attr  = torch.tensor(weights_bi, dtype=torch.float32).unsqueeze(1)

    return edge_index, edge_attr


# Строим WGCNA-граф для каждого timepoint-а
hdwgcna_edges_per_t = []
hdwgcna_attrs_per_t = []

for t in timepoints:
    mask = adata_hvg.obs['time'] == t
    X_t  = adata_hvg[mask].X
    if scipy.sparse.issparse(X_t):
        X_t = X_t.toarray()

    print(f"t={int(t):3d}  ({X_t.shape[0]} клеток) ...", end=" ", flush=True)
    ei, ea = build_wgcna_per_timepoint(X_t)
    hdwgcna_edges_per_t.append(ei)
    hdwgcna_attrs_per_t.append(ea)
    print(f"рёбер: {ei.shape[1]:,}")

print("\nГотово.")

## 11. Фичи генов per timepoint\n\nДля каждого гена и timepoint-а строим вектор признаков:\n- средняя экспрессия (по всем клеткам этого timepoint-а)\n- дисперсия экспрессии\n- доля ненулевых клеток (detection rate)\n\nЭто `gene_feats_per_t` — вход в модель.

In [ ]:
import scipy.sparse as sp

def gene_features(adata_t) -> np.ndarray:
    """
    Для одного timepoint-а → [n_genes, 3]:
        [mean_expr, var_expr, detection_rate]
    """
    X = adata_t.X
    if sp.issparse(X):
        X = X.toarray()
    mean  = X.mean(axis=0)                      # [n_genes]
    var   = X.var(axis=0)                       # [n_genes]
    detec = (X > 0).mean(axis=0)               # [n_genes]
    return np.stack([mean, var, detec], axis=1) # [n_genes, 3]


gene_feats_per_t = []

for t in timepoints:
    mask  = adata_hvg.obs['time'] == t
    feats = gene_features(adata_hvg[mask])
    gene_feats_per_t.append(torch.tensor(feats, dtype=torch.float32))
    print(f"t={int(t):3d}  gene_feats: {feats.shape}  mean_expr range: [{feats[:,0].min():.3f}, {feats[:,0].max():.3f}]")

GENE_FEAT_DIM = gene_feats_per_t[0].shape[1]   # = 3
print(f"\ngene_feat_dim = {GENE_FEAT_DIM}")

## 12. Запуск модели на реальных данных\n\nТеперь собираем всё вместе: инициализируем `GeneTrajectoryModel` с `gene_feat_dim=3` (реальные фичи вместо dummy) и прогоняем forward pass.

In [ ]:
N_GENES = len(gene_names)

model_real = GeneTrajectoryModel(
    gene_feat_dim = GENE_FEAT_DIM,
    hidden_dim    = 128,
    emb_dim       = 128,
    gat_layers    = 3,
    gat_heads     = 4,
    dropout       = 0.1,
)

print(f"Параметров в модели: {sum(p.numel() for p in model_real.parameters()):,}")

with torch.no_grad():
    out = model_real(
        gene_feats_per_t    = gene_feats_per_t,
        hdwgcna_edges_per_t = hdwgcna_edges_per_t,
        hdwgcna_attrs_per_t = hdwgcna_attrs_per_t,
    )

traj_emb = out["trajectory_emb"]
print(f"trajectory_emb : {traj_emb.shape}")
print(f"delta          : {out['delta'].shape}")
print(f"timepoints     : {len(out['snapshots'])}")

emb_df = pd.DataFrame(
    traj_emb.numpy(),
    index=gene_names,
    columns=[f"dim_{i}" for i in range(traj_emb.shape[1])]
)
print(f"\nПример — первые 3 гена:")
print(emb_df.iloc[:3, :5])

# ProgramEncoder
print("\n── ProgramEncoder (PPGN на ~200 генах программы) ──")
prog_encoder = ProgramEncoder(gene_feat_dim=GENE_FEAT_DIM)
N_PROG     = 200
prog_idx   = torch.randint(0, N_GENES, (N_PROG,))
prog_feats = gene_feats_per_t[0][prog_idx]
prog_adj   = (torch.rand(N_PROG, N_PROG) > 0.95).float()
prog_adj   = (prog_adj + prog_adj.T).clamp(0, 1)

with torch.no_grad():
    prog_emb = prog_encoder(prog_feats, prog_adj)
print(f"program_emb : {prog_emb.shape}")

## 13. Поиск соседей по trajectory embeddings

Cosine similarity в пространстве `trajectory_emb` — аналог того что делает приложение, но на новых траекторных эмбеддингах вместо GCN.

In [ ]:
import torch.nn.functional as F

# ── Нормализуем эмбеддинги для cosine similarity ────────────────
emb_norm = F.normalize(traj_emb, dim=-1)   # [n_genes, 128]

def get_neighbors(query_gene: str, top_k: int = 20):
    """
    Возвращает top-K соседей query_gene по cosine similarity
    в пространстве trajectory_emb.
    """
    if query_gene not in gene_names:
        print(f"Ген '{query_gene}' не найден в {len(gene_names)} генах.")
        return None

    idx = gene_names.index(query_gene)
    q   = emb_norm[idx]                              # [128]

    sims  = (emb_norm @ q).cpu().numpy()             # [n_genes]
    order = sims.argsort()[::-1]                     # убывающий порядок

    rows = []
    for rank, i in enumerate(order[:top_k + 1]):
        if gene_names[i] == query_gene:
            continue
        rows.append({
            "rank":       len(rows) + 1,
            "gene":       gene_names[i],
            "similarity": round(float(sims[i]), 4),
        })
        if len(rows) == top_k:
            break

    df = pd.DataFrame(rows)
    return df


# ── Примеры ─────────────────────────────────────────────────────
for query in ["FOXM1", "MKI67", "HSPA1B", "DNAJB1"]:
    df = get_neighbors(query, top_k=10)
    if df is not None:
        print(f"\n{'='*50}")
        print(f"  Top-10 соседей {query}  (trajectory GNN)")
        print(f"{'='*50}")
        print(df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
from umap import UMAP

# ── UMAP по trajectory_emb для визуализации ─────────────────────
print("Вычисляем UMAP на trajectory_emb...")
reducer = UMAP(n_neighbors=15, min_dist=0.3, random_state=42, verbose=False)
umap_traj = reducer.fit_transform(traj_emb.numpy())   # [3887, 2]

# Выделяем гены интереса
HIGHLIGHT = {
    "FOXM1":  "#e63946",
    "MKI67":  "#e63946",
    "HSPA1B": "#1a6faf",
    "DNAJB1": "#1a6faf",
    "BIRC5":  "#ff6600",
    "TOP2A":  "#ff6600",
}

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(umap_traj[:, 0], umap_traj[:, 1],
           s=1.5, alpha=0.3, c="#cccccc", linewidths=0, rasterized=True)

for gene, color in HIGHLIGHT.items():
    if gene in gene_names:
        i = gene_names.index(gene)
        ax.scatter(umap_traj[i, 0], umap_traj[i, 1],
                   s=80, c=color, zorder=5, edgecolors="white", linewidths=0.8)
        ax.annotate(gene, (umap_traj[i, 0], umap_traj[i, 1]),
                    fontsize=9, fontweight="bold", color=color,
                    xytext=(4, 4), textcoords="offset points")

ax.set_xlabel("UMAP 1", fontsize=11)
ax.set_ylabel("UMAP 2", fontsize=11)
ax.set_title("Trajectory GNN embeddings — UMAP\n(все 3887 генов, выделены ключевые)", fontsize=12)
ax.set_facecolor("white")
ax.spines[["top", "right"]].set_visible(False)
patches = [mpatches.Patch(color="#e63946", label="Proliferative (FOXM1/MKI67)"),
           mpatches.Patch(color="#1a6faf", label="Quiescent (HSPA1B/DNAJB1)"),
           mpatches.Patch(color="#ff6600", label="Cell cycle (BIRC5/TOP2A)")]
ax.legend(handles=patches, frameon=False, fontsize=9, loc="upper right")
plt.tight_layout()
plt.savefig("/Users/zeev/CardamomOT/my_project/Data/trajectory_gnn_umap.png", dpi=150, bbox_inches="tight")
plt.show()

## 15. Сохранение эмбеддингов для приложения

Сохраняем в формате совместимом с `gcn_gene_embeddings_clusters.csv` —  
тот же формат что использует приложение для retrieval.

In [ ]:
from sklearn.cluster import KMeans

# K-means кластеризация на trajectory embeddings
N_CLUSTERS = 20
print(f"K-means кластеризация ({N_CLUSTERS} кластеров)...")
km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
traj_clusters = km.fit_predict(traj_emb.numpy())
print(f"Кластеры: {pd.Series(traj_clusters).value_counts().sort_index().to_dict()}")

# Сохраняем в формат приложения: index=gene, dim_0..dim_127, cluster
OUT_PATH = DATA_PATH + "trajectory_gene_embeddings.csv"

save_df = pd.DataFrame(
    traj_emb.numpy(),
    index=gene_names,
    columns=[f"dim_{i}" for i in range(traj_emb.shape[1])],
)
save_df["cluster"] = traj_clusters
save_df.index.name = None

save_df.to_csv(OUT_PATH)
print(f"Сохранено: {OUT_PATH}")
print(f"  shape : {save_df.shape}")
print(f"  genes : {len(gene_names)}")
print(f"  dims  : {traj_emb.shape[1]}")

# Placeholder аннотации кластеров (заполнить через LLM позже)
ann_df = pd.DataFrame({
    "cluster": range(N_CLUSTERS),
    "label":   [f"Trajectory cluster {i}" for i in range(N_CLUSTERS)],
})
ann_df.to_csv(DATA_PATH + "trajectory_cluster_annotations.csv", index=False)

sum_df = pd.DataFrame({
    "cluster": range(N_CLUSTERS),
    "summary": [f"Trajectory GNN cluster {i} — annotation pending." for i in range(N_CLUSTERS)],
})
sum_df.to_csv(DATA_PATH + "trajectory_cluster_summaries.csv", index=False)

print("Сохранены placeholder аннотации кластеров.")
print("\nДля загрузки в приложение загрузи на HuggingFace:")
print("  trajectory_gene_embeddings.csv")
print("  trajectory_cluster_annotations.csv")
print("  trajectory_cluster_summaries.csv")